# Week 8 Evaluation Expansion
- Compute metrics beyond R²: MAPE and MdAPE. (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.median_absolute_error.html)
    - Why is it difficult to find things on MdAPE? (https://stats.stackexchange.com/questions/596324/is-median-absolute-percentage-error-useless)
- Summarize insights (which price bands performed better)

- I'll clean up the code from previous files and re-do it here for better readability (and convenience)
- Hyperparameter tuning (https://www.geeksforgeeks.org/machine-learning/sklearn-model-hyper-parameters-tuning/)

In [4]:
# Importing modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hyperparameter tuning modules
from sklearn.model_selection import GridSearchCV

# Regression modules
from sklearn.linear_model import LinearRegression
from sklearn.metrics import median_absolute_error, mean_absolute_percentage_error

# Tree modules
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# XGBoost
from xgboost import XGBRegressor
from scipy.stats import loguniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

# Data Wrangling
Import and split data properly

In [5]:
# Importing cleaned data
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"
data_location = f"{root}/IDX_Exchange/deliverables"

testing_set = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set.csv')
training_set = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set.csv') 

# feature engineered datasets
testing_set_fe = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set_fe.csv')
training_set_fe = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set_fe.csv') 

In [6]:
# Splitting training set into different months to test hyperparameter
months = ['2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2025-06']
training_set_split_by_months = []
for month in range(len(months)):
    training_set_split_by_months.append(training_set[training_set['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_split_by_months[month] = training_set_split_by_months[month].drop(columns='CloseDate')

# Get Target and Normal Vars from the Testing set
testing_vars = testing_set.drop(columns='ClosePrice')
testing_target = testing_set['ClosePrice']

In [7]:
# Do the same with featured engineered sets

training_set_months_fe = []
for month in range(len(months)):
    training_set_months_fe.append(training_set_fe[training_set_fe['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_months_fe[month] = training_set_months_fe[month].drop(columns='CloseDate')
    
testing_vars_fe = testing_set_fe.drop(columns='ClosePrice')
testing_target_fe = testing_set_fe['ClosePrice']

In [8]:
""" 
    Function to test different time frames of training data
    You can include up to 13 months
"""
def choose_test_months(num_months):
    if(num_months > 13 or num_months < 1):
        num_months = 13
        
    included_months = []
    for month in range(num_months):
        included_months.append(training_set_split_by_months[12 - month])
    
    return pd.concat(included_months, axis=0)

""" 
    Function to split trianing data into its variables and target
"""
def get_training_split(training_data):
    training_vars = training_data.drop(columns='ClosePrice')
    training_target = training_data['ClosePrice']
    
    return training_vars, training_target

In [9]:
# Creating dataframe used to store scores between models
evaluations = pd.DataFrame(columns=['Model', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])

# Baseline Model
- From 03_baseline_model.ipynb
- A simple regression model using sklearn
- Test out number of months as hyperparameter

In [10]:
def baseline_regression_model():
    # number of months
    num_months = [1, 6, 12, 13]
    
    regression_model_performance = pd.DataFrame(columns=['Num_Months', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])
    
    for num in num_months:
        training_data = choose_test_months(num_months=num)
        training_vars, training_target = get_training_split(training_data)
        
        # Create models
        model = LinearRegression().fit(training_vars, training_target)
        
        new_row = pd.DataFrame([{
            'Num_Months': num,
            'Training_R2': model.score(training_vars, training_target),
            'Training_MdAPE': median_absolute_error(training_target, model.predict(training_vars)),
            'Training_MAPE': mean_absolute_percentage_error(training_target, model.predict(training_vars)),
            'R2': model.score(testing_vars, testing_target),
            'MdAPE': median_absolute_error(testing_target, model.predict(testing_vars)),
            'MAPE': mean_absolute_percentage_error(testing_target, model.predict(testing_vars))
        }])

        regression_model_performance = pd.concat(
            [regression_model_performance, new_row],
            ignore_index=True
        )
        
        print(f"R2 of model trained on {num} months: {model.score(testing_vars, testing_target)}")

    
    regression_model_performance = regression_model_performance.sort_values(by='R2', ascending=False)
    print("Baseline Model Performance Summary:")    
    print(regression_model_performance)
    
    return regression_model_performance.iloc[0]
    
    
    

In [11]:
# Regress normal data
best_model = baseline_regression_model()

print("Baseline Regression Performance:")
print(f"Number of Months: {best_model['Num_Months']}")
print(f"Training R2: {best_model['Training_R2']}")
print(f"Training MdAPE: {best_model['Training_MdAPE']}")
print(f"Training MAPE: {best_model['Training_MAPE']}")
print(f"R2: {best_model['R2']}")
print(f"MdAPE: {best_model['MdAPE']}")
print(f"MAPE: {best_model['MAPE']}")

evaluations = pd.concat([evaluations, best_model], ignore_index=True)


C:\Users\donutii\AppData\Local\Temp\ipykernel_12544\1578255928.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  regression_model_performance = pd.concat(


R2 of model trained on 1 months: 0.4474074546268124
R2 of model trained on 6 months: 0.7018661811650542
R2 of model trained on 12 months: 0.7215168470647313
R2 of model trained on 13 months: 0.7215586516096621
Baseline Model Performance Summary:
  Num_Months  Training_R2  Training_MdAPE  Training_MAPE        R2  \
3         13     0.707523   190106.304610      13.891963  0.721559   
2         12     0.706574   189804.003664      13.893092  0.721517   
1          6     0.720403   185537.442318      14.667433  0.701866   
0          1     0.726964   184137.546210      13.865004  0.447407   

           MdAPE       MAPE  
3  189714.922489  12.060183  
2  189362.916179  12.031524  
1  184146.286611  12.381862  
0  422936.541098  22.311199  
Baseline Regression Performance:
Number of Months: 13
Training R2: 0.7075231518539578
Training MdAPE: 190106.30460956995
Training MAPE: 13.891962651288027
R2: 0.7215586516096621
MdAPE: 189714.92248880025
MAPE: 12.06018322331926


In [12]:
# Regress feature engineered data

# Regression Decision Tree
- From 04_model_comparison.ipynb
- DecisionTree using sklearn
- DecisionTrees are also relatively cheap in processing so I will use grid search here as well

In [13]:
def decisiontree_regression_model(num_months):
    params = {
        "criterion" : ['squared_error', 'friedman_mse', 'poisson'],
        "max_depth" : [20, 30, 40, 60]
    }
    
    training_data = choose_test_months(num_months)
    training_vars, training_target = get_training_split(training_data)
    
    # Create a GridSearch CV
    model_grid_searchcv = GridSearchCV(DecisionTreeRegressor(), params, cv=5, 
                                       scoring={'R2': 'r2', 'MdAPE': 'neg_median_absolute_error', 'MAPE': 'neg_mean_absolute_percentage_error'}, 
                                       refit='R2', n_jobs=-1)
    
    # Fit to training data
    model_grid_searchcv.fit(training_vars, training_target)
    
    # create cv
    cv_results = pd.DataFrame(model_grid_searchcv.cv_results_)
    cv_results["R2"] = -cv_results["mean_test_R2"]
    cv_results["MdAPE"] = -cv_results["mean_test_MdAPE"]
    cv_results["MAPE"] = cv_results["mean_test_MAPE"]
    
    print(f"Decision Tree Regression Model Performance for {num_months} months:")
    print(cv_results)
    
    # return the best model
    return model_grid_searchcv.best_estimator_

In [14]:
# regress normal data

model_aggregates = pd.DataFrame(columns=['Num_Months', 'Model', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])
num_months = [1, 6, 12, 13]
for months in num_months:
    training_data = choose_test_months(num_months=months)
    training_vars, training_target = get_training_split(training_data)
    
    best_model = decisiontree_regression_model(months)
    
    new_row = pd.DataFrame([{
            'Num_Months': months,
            'Model': f'DecisionTreeRegressor_{months}months',
            'Training_R2': best_model.score(training_vars, training_target),
            'Training_MdAPE': median_absolute_error(training_target, best_model.predict(training_vars)),
            'Training_MAPE': mean_absolute_percentage_error(training_target, best_model.predict(training_vars)),
            'R2': best_model.score(testing_vars, testing_target),
            'MdAPE': median_absolute_error(testing_target, best_model.predict(testing_vars)),
            'MAPE': mean_absolute_percentage_error(testing_target, best_model.predict(testing_vars))
        }])
    model_aggregates = pd.concat([model_aggregates, new_row], ignore_index=True)

print("Best model from different windows of training data:")    
model_aggregates = model_aggregates.sort_values(by='R2', ascending=False)
print(model_aggregates)    
evaluations = pd.concat([evaluations, model_aggregates.iloc[0]], ignore_index=True)

Decision Tree Regression Model Performance for 1 months:
    mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0        0.239220      0.029540         0.005812        0.002483   
1        0.275539      0.026237         0.004816        0.000920   
2        0.250833      0.019767         0.004190        0.000372   
3        0.244853      0.009654         0.004109        0.000124   
4        0.236438      0.010984         0.005781        0.003938   
5        0.247261      0.008643         0.004554        0.001085   
6        0.240206      0.011710         0.004379        0.000321   
7        0.266230      0.016091         0.005092        0.000875   
8        0.295157      0.030663         0.004115        0.000473   
9        0.296682      0.021942         0.004090        0.000582   
10       0.286505      0.018602         0.004088        0.000755   
11       0.255975      0.015270         0.003028        0.000260   

   param_criterion  param_max_depth  \
0    squared_error 

C:\Users\donutii\AppData\Local\Temp\ipykernel_12544\3802754821.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  model_aggregates = pd.concat([model_aggregates, new_row], ignore_index=True)


Decision Tree Regression Model Performance for 6 months:
    mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0        1.100422      0.024919         0.009297        0.001257   
1        1.299116      0.018861         0.014427        0.004128   
2        1.337295      0.046802         0.012580        0.002293   
3        1.329718      0.042950         0.012141        0.002755   
4        1.132326      0.016099         0.009777        0.002431   
5        1.329724      0.037867         0.015220        0.004391   
6        1.337662      0.058952         0.012127        0.002670   
7        1.322664      0.012891         0.010471        0.001118   
8        1.306265      0.018781         0.009808        0.001326   
9        1.523542      0.063366         0.009406        0.000551   
10       1.540182      0.024742         0.008503        0.000543   
11       1.438084      0.017606         0.007424        0.000951   

   param_criterion  param_max_depth  \
0    squared_error 

# Random Forest Regressor
- Also from 04_model_comparison.ipynb
- Using randomsearch because processing these takes quite a while

In [22]:

def randomforest_regression_model(num_months):
    params = {
        "criterion" : ['squared_error', 'friedman_mse', 'poisson'],
        "max_depth" : [20, 30, 40, 60],
        "max_features": ['sqrt', 'log2', None],
        "random_state": [20, 42, 60]
        # TODO: get more params here
    }
    
    training_data = choose_test_months(num_months)
    training_vars, training_target = get_training_split(training_data)
    
    # Create a Random Search CV
    model_rand_searchcv = RandomizedSearchCV(RandomForestRegressor(), params, cv=5, 
                                            scoring={'R2': 'r2', 'MdAPE': 'neg_median_absolute_error', 'MAPE': 'neg_mean_absolute_percentage_error'}, 
                                            refit='R2', n_jobs=-1)
    # fit to data
    model_rand_searchcv.fit(training_vars, training_target)
    
    # create cv
    cv_results = pd.DataFrame(model_rand_searchcv.cv_results_)
    cv_results["R2"] = -cv_results["mean_test_R2"]
    cv_results["MdAPE"] = -cv_results["mean_test_MdAPE"]
    cv_results["MAPE"] = cv_results["mean_test_MAPE"]
    
    print(f"Random Forest Regression Model Performance for {num_months} months:")
    print(cv_results)
    
    # return the best model
    return model_rand_searchcv.best_estimator_

In [23]:
# regress normal data

model_aggregates = pd.DataFrame(columns=['Num_Months', 'Model', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])
num_months = [6, 12, 13]
for months in num_months:
    training_data = choose_test_months(num_months=months)
    training_vars, training_target = get_training_split(training_data)
    
    best_model = randomforest_regression_model(months)
    
    new_row = pd.DataFrame([{
            'Num_Months': months,
            'Model': f'DecisionTreeRegressor_{months}months',
            'Training_R2': best_model.score(training_vars, training_target),
            'Training_MdAPE': median_absolute_error(training_target, best_model.predict(training_vars)),
            'Training_MAPE': mean_absolute_percentage_error(training_target, best_model.predict(training_vars)),
            'R2': best_model.score(testing_vars, testing_target),
            'MdAPE': median_absolute_error(testing_target, best_model.predict(testing_vars)),
            'MAPE': mean_absolute_percentage_error(testing_target, best_model.predict(testing_vars))
        }])
    model_aggregates = pd.concat([model_aggregates, new_row], ignore_index=True)

print("Best model from different windows of training data:")    
model_aggregates = model_aggregates.sort_values(by='R2', ascending=False)
print(model_aggregates)    
evaluations = pd.concat([evaluations, model_aggregates.iloc[0]], ignore_index=True)

Random Forest Regression Model Performance for 6 months:
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0      16.712568      0.312404         0.490740        0.030032   
1      84.098532      1.711687         0.460920        0.015848   
2      83.310880      0.753506         0.476248        0.022371   
3      73.276365      0.686724         0.371155        0.011474   
4      16.248405      0.320637         0.438201        0.015430   
5      83.863848      1.041134         0.455945        0.020051   
6      19.355481      0.377142         0.450224        0.020296   
7      16.161082      0.190156         0.457028        0.021588   
8      82.719884      0.905900         0.339036        0.009922   
9      16.059386      0.139973         0.420652        0.019648   

   param_random_state param_max_features  param_max_depth param_criterion  \
0                  20               log2               40    friedman_mse   
1                  60               None           

C:\Users\donutii\AppData\Local\Temp\ipykernel_12544\3725058193.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  model_aggregates = pd.concat([model_aggregates, new_row], ignore_index=True)


Random Forest Regression Model Performance for 12 months:
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0      49.260617      0.826268         3.660931        0.486287   
1      50.013864      0.949466         3.489142        0.416286   
2     198.057838      2.414824         0.897542        0.025327   
3      54.380012      0.492235         1.693100        0.037622   
4      44.505069      0.798159         1.547911        0.097141   
5      52.885110      0.836328         1.437108        0.061296   
6     234.749355      2.046362         1.611466        0.154725   
7     192.304239      1.277330         0.960513        0.064947   
8     244.884574     10.618251         1.187236        0.115375   
9     232.857269      1.145099         0.947065        0.056859   

   param_random_state param_max_features  param_max_depth param_criterion  \
0                  60               log2               60   squared_error   
1                  60               sqrt          

# Histrogram Boost
- From 05_advanced_models.ipynb
- A Gradient Boosted regressor optimized for large sample sizes. From scikit, so is compatible with randomized search cv

In [26]:
def histboost_regression_model(num_months):
    params = {
        "max_iter": [100, 300, 500, 600, 1000],
        "max_leaf_nodes": [5, 10, 20, 50, 75, 100],
        "learning_rate": loguniform(0.01, 1),
    }
    
    training_data = choose_test_months(num_months)
    training_vars, training_target = get_training_split(training_data)
    
    # Create a GridSearch CV
    model_rand_searchcv = RandomizedSearchCV(HistGradientBoostingRegressor(), params, cv=5, 
                                                scoring={'R2': 'r2', 'MdAPE': 'neg_median_absolute_error', 'MAPE': 'neg_mean_absolute_percentage_error'}, 
                                                refit='R2', n_jobs=-1)
    # fit to data
    model_rand_searchcv.fit(training_vars, training_target)
    
    # create cv
    cv_results = pd.DataFrame(model_rand_searchcv.cv_results_)
    cv_results["R2"] = -cv_results["mean_test_R2"]
    cv_results["MdAPE"] = -cv_results["mean_test_MdAPE"]
    cv_results["MAPE"] = cv_results["mean_test_MAPE"]
    
    print(f"Random Forest Regression Model Performance for {num_months} months:")
    print(cv_results)
    
    # return the best model
    return model_rand_searchcv.best_estimator_

In [27]:
# Regress normal data
model_aggregates = pd.DataFrame(columns=['Num_Months', 'Model', 'Training_R2', 'Training_MdAPE', 'Training_MAPE', 'R2', 'MdAPE', 'MAPE'])
num_months = [6, 12, 13]
for months in num_months:
    training_data = choose_test_months(num_months=months)
    training_vars, training_target = get_training_split(training_data)
    
    best_model = histboost_regression_model(months)
    
    new_row = pd.DataFrame([{
            'Num_Months': months,
            'Model': f'DecisionTreeRegressor_{months}months',
            'Training_R2': best_model.score(training_vars, training_target),
            'Training_MdAPE': median_absolute_error(training_target, best_model.predict(training_vars)),
            'Training_MAPE': mean_absolute_percentage_error(training_target, best_model.predict(training_vars)),
            'R2': best_model.score(testing_vars, testing_target),
            'MdAPE': median_absolute_error(testing_target, best_model.predict(testing_vars)),
            'MAPE': mean_absolute_percentage_error(testing_target, best_model.predict(testing_vars))
        }])
    model_aggregates = pd.concat([model_aggregates, new_row], ignore_index=True)

print("Best model from different windows of training data:")    
model_aggregates = model_aggregates.sort_values(by='R2', ascending=False)
print(model_aggregates)    
evaluations = pd.concat([evaluations, model_aggregates.iloc[0]], ignore_index=True)

Random Forest Regression Model Performance for 6 months:
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       4.112887      0.120562         0.225942        0.020905   
1       7.716900      0.533074         0.573656        0.111735   
2       1.973680      0.303023         0.189583        0.026949   
3       6.486901      1.044716         0.576822        0.088081   
4       1.713908      0.107324         0.182172        0.010736   
5      13.687121      1.224879         1.061194        0.138095   
6       3.374019      0.783771         0.351342        0.092182   
7       7.303054      0.905305         0.708417        0.087979   
8       3.221627      0.679205         0.339493        0.089158   
9       6.825495      0.647125         0.522065        0.013814   

   param_learning_rate  param_max_iter  param_max_leaf_nodes  \
0             0.033758             100                    50   
1             0.110132            1000                   100   
2            

C:\Users\donutii\AppData\Local\Temp\ipykernel_12544\3071975448.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  model_aggregates = pd.concat([model_aggregates, new_row], ignore_index=True)


Random Forest Regression Model Performance for 12 months:
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0      39.576610      1.935351         4.119468        0.182397   
1       7.844256      0.889547         0.833883        0.148884   
2       8.261718      1.305236         0.908470        0.208356   
3       3.285757      0.518699         0.317929        0.054059   
4       4.809272      0.621124         0.434448        0.044721   
5       3.532706      0.281723         0.398975        0.065373   
6      23.391820      0.681879         1.808933        0.125303   
7       8.237410      0.647464         1.003364        0.106493   
8       8.012883      0.609032         0.898642        0.100363   
9      12.401940      0.528968         0.999494        0.043909   

   param_learning_rate  param_max_iter  param_max_leaf_nodes  \
0             0.039860            1000                    50   
1             0.354165             500                     5   
2           

# Consolidate evaluations
- Ideally add the best of each model to a csv and then export it

In [20]:
# Consolidate and export evaluations here